# **MODULO 1: Analissi de mercado musical para el artista**


* **Datos:**
    * Fuente de datos: Last.fm API (consumo real de usuarios)
    * Datos objetivo: Streams por país, Género musical, Playlist placement, TikTok trends, Crecimiento de artistas similares...
    * EXTRA DATOS: EDAD DE LA AUDIENCIA PARA AVERIGUAR TARGET SEGÚN PLATAFORMA. (MÉTRICAS AUDIENCIA TIK TOK)

* **Análisis (EDA):** crecimiento por género, países con más crecimiento, correlación entre playlist y streams, estacionalidad.
    * Feature Engineering: Ratio energía/valence, duración normalizada, explicit flag, features por artista

* **ML:** Predicción de streams de una canción, Predicción de viralidad, 
    * Modelos: Random Forest, XGBoost, LSTM para series temporales

* Output: “Tu canción tiene 65% probabilidad de viralizarse en México”, “Este beat funciona mejor en playlist de chill trap”





### **ÍNDICE DE IMPRESCINDIBLES PARA MODULO 1**



* **Módulo es el core estratégico: entender el mercado + base para predecir hits.**


* **DATOS: Se usan los datos de Last.fm API ya que  se basan en consumo real de usuarios.**

### Imports

In [2]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import requests
 


# **DATA**


## Imports y Set up

In [3]:
import time
import requests
import pandas as pd
import os 

## Recopilación de datos en Last.fm API

### Endpoints & Pipeline API

| Endpoint             | Devuelve     |
| -------------------- | ---------------- |
| `chart.getTopTracks` | top global       |
| `geo.getTopTracks`   | top por país     |
| `tag.getTopTracks`   | top por género   |

Pipeline API:
1. Llamas a endpoints
2. Controlas velocidad (throttling)
3. Evitas errores (retry + backoff)
4. Añades margen de seguridad
5. Combinas datos

* Conceptos:
    * Throttling → ir lento a propósito
    * Margen de seguridad → no ir al límite
    * Retry → reintentar si falla
    * Backoff → esperar más cada vez
    * Endpoint → tipo de datos que pides

In [4]:

API_KEY = '63e059c3c912a3f642daf2372484d183'
BASE_URL = 'http://ws.audioscrobbler.com/2.0/'
DELAY = 0.5  


### Data base para el análisis:

* Función data TopTracks

In [5]:
def fetch_top_tracks(page=1, limit=200, retries=5):
    params = {
        'method': 'chart.getTopTracks',
        'api_key': API_KEY,
        'format': 'json',
        'page': page,
        'limit': limit
    }

    for attempt in range(retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=10)

            if response.status_code in [429, 500, 502, 503]:
                wait_time = 2 ** attempt
                print(f"[{response.status_code}] Retry en {wait_time}s...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")
            time.sleep(2 ** attempt)

    print(f"❌ Fallo total en página {page}")
    return None

In [6]:
fetch_top_tracks()

{'tracks': {'track': [{'name': 'Stateside + Zara Larsson',
    'duration': '176',
    'playcount': '15257145',
    'listeners': '1034823',
    'mbid': 'ffbf7862-2476-4164-ac32-f5904ccefe0f',
    'url': 'https://www.last.fm/music/PinkPantheress/_/Stateside+%252B+Zara+Larsson',
    'streamable': {'#text': '0', 'fulltrack': '0'},
    'artist': {'name': 'PinkPantheress',
     'mbid': '7441014f-f8f5-494f-81db-ff166fbc078d',
     'url': 'https://www.last.fm/music/PinkPantheress'},
    'image': [{'#text': 'https://lastfm.freetls.fastly.net/i/u/34s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'small'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/64s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'medium'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/174s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'large'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'extralarge'}]},
   {'name': 'B

* No repetir datos del csv que ya tenemos

In [7]:
if os.path.exists("backup_manual.csv"):
    df_existing = pd.read_csv("backup_manual.csv")
else:
    df_existing = pd.DataFrame()

/tmp/ipykernel_3735/560536097.py:2: DtypeWarning: Columns (0: streamable, 1: image) have mixed types. Specify dtype option on import or set low_memory=False.
  df_existing = pd.read_csv("backup_manual.csv")


In [8]:
# crea conjunto de canciones existentes:
existing_tracks = set()

if not df_existing.empty:
    for _, row in df_existing.iterrows():
        key = f"{row['name']}_{row['artist']}"
        existing_tracks.add(key)

In [15]:
df_existing

,name,duration,playcount,listeners,mbid,url,streamable,artist,image,country,genre_tag
0,Stateside + Zara Larsson,176,14787193.0,1013640.0,ffbf7862-2476-4164-ac32-f5904ccefe0f,https://www.last.fm/music/PinkPantheress/_/Sta...,"{'#text': '0', 'fulltrack': '0'}","{'name': 'PinkPantheress', 'mbid': '7441014f-f...",[{'#text': 'https://lastfm.freetls.fastly.net/...,NaN,NaN
1,Body to Body,189,2615165.0,250542.0,6dfdfc61-89ff-451a-a5e9-5319065beae7,https://www.last.fm/music/BTS/_/Body+to+Body,"{'#text': '0', 'fulltrack': '0'}","{'name': 'BTS', 'mbid': '0d79fe8e-ba27-4859-bb...",[{'#text': 'https://lastfm.freetls.fastly.net/...,NaN,NaN
2,Hooligan,182,2367820.0,233389.0,a045f252-ea1a-42c3-b7a5-f69979e39394,https://www.last.fm/music/BTS/_/Hooligan,"{'#text': '0', 'fulltrack': '0'}","{'name': 'BTS', 'mbid': '0d79fe8e-ba27-4859-bb...",[{'#text': 'https://lastfm.freetls.fastly.net/...,NaN,NaN
3,Swim,159,6050938.0,231653.0,6f33dc05-cdc0-4a2f-8039-e8fed082eec6,https://www.last.fm/music/BTS/_/Swim,"{'#text': '0', 'fulltrack': '0'}","{'name': 'BTS', 'mbid': '0d79fe8e-ba27-4859-bb...",[{'#text': 'https://lastfm.freetls.fastly.net/...,NaN,NaN
4,FYA,180,2324099.0,227425.0,802e86e2-5da4-40c5-a265-933eb64d864b,https://www.last.fm/music/BTS/_/FYA,"{'#text': '0', 'fulltrack': '0'}","{'name': 'BTS', 'mbid': '0d79fe8e-ba27-4859-bb...",[{'#text': 'https://lastfm.freetls.fastly.net/...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
100223,Sleeping Heart,208,NaN,NaN,69be36b6-82e5-46cf-b8eb-dbca9053b066,https://www.last.fm/music/Giles+Corey/_/Sleepi...,NaN,Giles Corey,NaN,TAG,singer-songwriter
100224,Come in with the Rain,238,NaN,NaN,02e8528b-aae1-410d-b686-b737ffce691a,https://www.last.fm/music/Taylor+Swift/_/Come+...,NaN,Taylor Swift,NaN,TAG,singer-songwriter
100225,Maybe Angels,371,NaN,NaN,0366e72a-97eb-367f-895c-c29e707a3801,https://www.last.fm/music/Sheryl+Crow/_/Maybe+...,NaN,Sheryl Crow,NaN,TAG,singer-songwriter
100226,Shiver Me Timbers,261,NaN,NaN,0214c6c4-9f53-369b-acaf-8518a9b9ad25,https://www.last.fm/music/Tom+Waits/_/Shiver+M...,NaN,Tom Waits,NaN,TAG,singer-songwriter


* Batch & DataFrame para TopTracks

In [9]:
all_tracks = []

for page in range(1, 50):  # empieza pequeño

    data = fetch_top_tracks(page=page)

    if data is None:
        print(f"⚠️ Saltando página {page}")
        time.sleep(5)
        continue

    tracks = data.get('tracks', {}).get('track', [])

    for track in tracks:
        name = track.get('name')
        artist = track.get('artist', {}).get('name')

        key = f"{name}_{artist}"

        # Evitar duplicados
        if key in existing_tracks:
            continue

        all_tracks.append({
            'name': name,
            'artist': artist,
            'playcount': track.get('playcount'),
            'listeners': track.get('listeners'),
            'mbid': track.get('mbid'),
            'duration': track.get('duration'),
            'url': track.get('url')
        })

        existing_tracks.add(key)

    time.sleep(DELAY)

    # Guardado frecuente
    if page % 10 == 0:
        df_temp = pd.DataFrame(all_tracks)
        df_total = pd.concat([df_existing, df_temp], ignore_index=True)
        df_total.to_csv("backup_manual.csv", index=False)
        print(f"Guardado en página {page}")

Guardado en página 10
Guardado en página 20
Guardado en página 30
Guardado en página 40


* Guardado frecuente

In [10]:
df_new = pd.DataFrame(all_tracks)
df_final = pd.concat([df_existing, df_new], ignore_index=True)

df_final.drop_duplicates(subset=['name', 'artist'], inplace=True)

df_final.to_csv("backup_manualt.csv", index=False)

print("Recolección completada")

Recolección completada


In [11]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 100341 entries, 0 to 100340
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   name        100341 non-null  str   
 1   duration    100341 non-null  object
 2   playcount   19912 non-null   object
 3   listeners   25126 non-null   object
 4   mbid        89345 non-null   str   
 5   url         100341 non-null  str   
 6   streamable  9999 non-null    str   
 7   artist      100339 non-null  str   
 8   image       9999 non-null    str   
 9   country     80429 non-null   str   
 10  genre_tag   75215 non-null   str   
dtypes: object(3), str(8)
memory usage: 8.4+ MB


#### Erroes API 23-03-2026:

* Errores en peticiones:

| Código | Significado | Quién tiene el problema     |
| ------ | ----------- | --------------------------- |
| 403    | Forbidden   | ❌ tú (API key, permisos)    |
| 429    | Rate limit  | ⚠️ tú (demasiadas requests) |
| 502    | Bad Gateway | ❌ servidor (Last.fm)        |

* Errores después de obtener data: 
    - Bug en menos de 10K datos
    - Solucionar problema y no volver a guardar data duplicada
    - Descarga de datos sigue siendo insuficiente. Tenemos menos de 20K y necesitamos 60K. (Error 2 en doc ERRORS -> Solución: Combinar fuentes).

* Solucionar ERROR 2: descarga de data de más generos y países para completar las filas faltantes con información de valores para el análisis.


### Data por países:

* Función data países:

In [12]:
def fetch_geo_tracks(country, page=1, limit=200, retries=5):
    params = {
        'method': 'geo.getTopTracks',
        'country': country,
        'api_key': API_KEY,
        'format': 'json',
        'page': page,
        'limit': limit
    }

    for attempt in range(retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=10)

            if response.status_code in [429, 500, 502, 503]:
                time.sleep(2 ** attempt)
                continue

            response.raise_for_status()
            return response.json()

        except:
            time.sleep(2 ** attempt)

    return None

* Lista por paises:

In [13]:
countries = ['Spain', 'United States', 'United Kingdom', 'Brazil', 'Germany', 'France', 'Mexico', 'Peru', 'Japan', 'Netherlands']

for country in countries:
    print(f"🌍 País: {country}")

    for page in range(1, 20):

        data = fetch_geo_tracks(country, page)

        if data is None:
            continue

        tracks = data.get('tracks', {}).get('track', [])

        for track in tracks:
            name = track.get('name')
            artist = track.get('artist', {}).get('name')

            key = f"{name}_{artist}"

            if key in existing_tracks:
                continue

            all_tracks.append({
                'name': name,
                'artist': artist,
                'playcount': track.get('playcount'),
                'listeners': track.get('listeners'),
                'mbid': track.get('mbid'),
                'duration': track.get('duration'),
                'url': track.get('url'),
                'country': country
            })

            existing_tracks.add(key)

        time.sleep(DELAY)

🌍 País: Spain
🌍 País: United States
🌍 País: United Kingdom
🌍 País: Brazil


KeyboardInterrupt: 

In [ ]:
FILE_NAME = 'backup_manual.csv'
df_new = pd.DataFrame(all_tracks)
df_final = pd.concat([df_existing, df_new], ignore_index=True)

df_final.drop_duplicates(subset=['name', 'artist'], inplace=True)

df_final.to_csv(FILE_NAME, index=False)

print("Datos de géneros guardados")

Datos de géneros guardados


### Data por género musical:

In [ ]:
def fetch_tag_tracks(tag, page=1, limit=200, retries=5):
    params = {
        'method': 'tag.getTopTracks',
        'tag': tag,
        'api_key': API_KEY,
        'format': 'json',
        'page': page,
        'limit': limit
    }

    for attempt in range(retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=10)

            if response.status_code in [429, 500, 502, 503]:
                wait_time = 2 ** attempt
                print(f"[{response.status_code}] Retry en {wait_time}s...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")
            time.sleep(2 ** attempt)

    print(f"Fallo en tag {tag}, página {page}")
    return None

In [ ]:
def get_top_tags():
    params = {
        'method': 'tag.getTopTags',
        'api_key': API_KEY,
        'format': 'json'
    }

    response = requests.get(BASE_URL, params=params)
    data = response.json()

    tags = [tag['name'] for tag in data['toptags']['tag']]
    return tags

tags = get_top_tags()
print(tags[:21])

NameError: name 'requests' is not defined

In [ ]:
tags = ['rock', 'electronic', 'seen live', 'alternative', 'pop', 'indie', 'female vocalists', 'metal', 'alternative rock', 'jazz', 'classic rock', 'ambient', 'experimental', 'folk', 'indie rock', 'punk', 'Hip-Hop', 'hard rock', 'black metal', 'instrumental', 'singer-songwriter']

for tag in tags:
    print(f"🎧 Género: {tag}")

    for page in range(1, 20):

        data = fetch_tag_tracks(tag, page)

        if data is None:
            continue

        tracks = data.get('tracks', {}).get('track', [])

        for track in tracks:
            name = track.get('name')
            artist = track.get('artist', {}).get('name')

            key = f"{name}_{artist}"

            # 🚫 evitar duplicados
            if key in existing_tracks:
                continue

            all_tracks.append({
                'name': name,
                'artist': artist,
                'playcount': track.get('playcount'),
                'listeners': track.get('listeners'),
                'mbid': track.get('mbid'),
                'duration': track.get('duration'),
                'url': track.get('url'),
                'country': 'TAG',
                'genre_tag': tag
            })

            existing_tracks.add(key)

        time.sleep(DELAY)

    print(f"Completado género: {tag}")

🎧 Género: rock


NameError: name 'requests' is not defined

In [ ]:
df_new = pd.DataFrame(all_tracks)
df_final = pd.concat([df_existing, df_new], ignore_index=True)

df_final.drop_duplicates(subset=['name', 'artist'], inplace=True)

df_final.to_csv(FILE_NAME, index=False)

print("Datos de géneros guardados")

NameError: name 'all_tracks' is not defined

In [ ]:
import pandas as pd
df_final = pd.read_csv('backup_manual.csv')

/tmp/ipykernel_1294/1401388933.py:2: DtypeWarning: Columns (0: streamable, 1: image) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv('backup_manual.csv')


In [ ]:
df_final.shape

(100228, 11)

# **EDA**

1. **Análisis del mercado musical (descriptivo):** 

(1) Top canciones / artistas por periodo (tiempo); 

(2) Evolución temporal de popularidad; 

(3) Géneros en crecimiento vs decrecimiento

2. **Análisis por género:**

* Distribución de features por género: danceability, energy, valence, tempo

* ¿Qué hace único a cada género?

3. **Análisis geográfico** 

(1) Países con mayor consumo; 

(2) Diferencias culturales en features.

4. **Ciclos de éxito:** 

(1) Duración de popularidad de una canción; 

(2) Tiempo entre picos de streams; 

(3) Estacionalidad (ej: “hits de verano”)

**Features engineering (clave para predecir un hit):**


1. **Correlación con popularidad:** Qué variables impactan más: danceability ↑, energy ↑, valence (positivo vs triste)

* Heatmap de correlaciones

2. **Ingeniería de variables:** (1) Ratio energía/valence; (2) Duración normalizada; (3) Explicit vs no explicit; (4) Features agregadas por artista

3. **Segmentación de canciones:** 

* Clustering para KMeans
* Tipos de canciones: “club hit”, “chill”, “sad viral”,... 


## Análisis

### 0. **Información y limpieza del DataFrame**

#### Información general:

In [ ]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 100228 entries, 0 to 100227
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   name        100228 non-null  str    
 1   duration    100228 non-null  int64  
 2   playcount   19799 non-null   float64
 3   listeners   25013 non-null   float64
 4   mbid        89266 non-null   str    
 5   url         100228 non-null  str    
 6   streamable  9999 non-null    str    
 7   artist      100226 non-null  str    
 8   image       9999 non-null    str    
 9   country     80429 non-null   str    
 10  genre_tag   75215 non-null   str    
dtypes: float64(2), int64(1), str(8)
memory usage: 8.4 MB


In [ ]:
df_final.head()

,name,duration,playcount,listeners,mbid,url,streamable,artist,image,country,genre_tag
0,Stateside + Zara Larsson,176,14787193.0,1013640.0,ffbf7862-2476-4164-ac32-f5904ccefe0f,https://www.last.fm/music/PinkPantheress/_/Sta...,"{'#text': '0', 'fulltrack': '0'}","{'name': 'PinkPantheress', 'mbid': '7441014f-f...",[{'#text': 'https://lastfm.freetls.fastly.net/...,NaN,NaN
1,Body to Body,189,2615165.0,250542.0,6dfdfc61-89ff-451a-a5e9-5319065beae7,https://www.last.fm/music/BTS/_/Body+to+Body,"{'#text': '0', 'fulltrack': '0'}","{'name': 'BTS', 'mbid': '0d79fe8e-ba27-4859-bb...",[{'#text': 'https://lastfm.freetls.fastly.net/...,NaN,NaN
2,Hooligan,182,2367820.0,233389.0,a045f252-ea1a-42c3-b7a5-f69979e39394,https://www.last.fm/music/BTS/_/Hooligan,"{'#text': '0', 'fulltrack': '0'}","{'name': 'BTS', 'mbid': '0d79fe8e-ba27-4859-bb...",[{'#text': 'https://lastfm.freetls.fastly.net/...,NaN,NaN
3,Swim,159,6050938.0,231653.0,6f33dc05-cdc0-4a2f-8039-e8fed082eec6,https://www.last.fm/music/BTS/_/Swim,"{'#text': '0', 'fulltrack': '0'}","{'name': 'BTS', 'mbid': '0d79fe8e-ba27-4859-bb...",[{'#text': 'https://lastfm.freetls.fastly.net/...,NaN,NaN
4,FYA,180,2324099.0,227425.0,802e86e2-5da4-40c5-a265-933eb64d864b,https://www.last.fm/music/BTS/_/FYA,"{'#text': '0', 'fulltrack': '0'}","{'name': 'BTS', 'mbid': '0d79fe8e-ba27-4859-bb...",[{'#text': 'https://lastfm.freetls.fastly.net/...,NaN,NaN


> **Observaciones:**  
> * Tenemos un total de 100228 filas y 11 columnas:  
>   - Columnas numéricas: duration, playcount, listeners (variables clave)  
>   - Columnas string: name, image, country, streamable,genre_tag (etiquetas)  
>   - Columnas identificadoras: url, mbid (código identificador)
> * Valores nulos: Las filas que contienen más valores nulos son playcount, listeners y mbid. 


**Descripción de variables del dataset:** 


>**Variables principales:**
> 
> * **name** Nombre de la canción. Identificador principal del track.
> 
> * **artist** Nombre del artista. Permite agrupar por creador y analizar popularidad.
> 
> * **playcount** Número total de reproducciones. (Indicador directo de popularidad global)
> 
> * **listeners** Número de usuarios únicos que han escuchado la canción. (Permite medir alcance real (no solo repeticiones))
> 
> * **duration** Duración de la canción en milisegundos.(Útil para analizar tendencias (ej: canciones cortas tipo TikTok))

---

> **Variables contextuales**
> 
> * **country** País desde donde se recoge el dato. (Permite análisis geográfico del mercado musical.)
> 
> * **genre_tag** Género musical asociado (tag de Last.fm). (Permite segmentar por estilo musical.)

---

> **Variables técnicas**
> 
> * **mbid** Identificador único de MusicBrainz. (Sirve para eliminar duplicados con precisión.)
> 
> * **url** Enlace a la canción en Last.fm. (No aporta valor analítico directo.)

---

> **Variables descartadas**
> 
> * **streamable** Indica si la canción es reproducible. (No aporta valor para análisis de popularidad.)
> 
> * **image** Imagen del track/artista. (Irrelevante para análisis numérico.)

---



#### Limpieza de datos:

* Duplicados:


In [ ]:
df_final.drop_duplicates(subset=['name', 'artist'], inplace=True)

print("Después de duplicados:", df_final.shape)

Después de duplicados: (100228, 11)


* Valores nulos:

In [ ]:
df_final.isnull().sum()

name              0
duration          0
playcount     80429
listeners     75215
mbid          10962
url               0
streamable    90229
artist            2
image         90229
country       19799
genre_tag     25013
dtype: int64

In [ ]:
df_clean = df_final.dropna(subset=['playcount', 'listeners']).copy()

print("Dataset limpio:", df_clean.shape)

Dataset limpio: (19799, 11)


* Se eliminan las columnas que no son relevantes para el análisis:

In [ ]:
df_music = df_clean.drop(['image','url','mbid'],axis=1).copy()

* Se rellenan los valores nulos en country y genre_tag:

In [ ]:
df_music['country'] = df_clean['country'].fillna('UNKNOWN')
df_music['genre_tag'] = df_clean['genre_tag'].fillna('UNKNOWN')

* En algunas filas se encuentra información guardada en diccionarios y solo queremos visualizar un dato:


In [ ]:
df_clean['artist'][0]

"{'name': 'PinkPantheress', 'mbid': '7441014f-f8f5-494f-81db-ff166fbc078d', 'url': 'https://www.last.fm/music/PinkPantheress'}"

DUDA, ME HE QUEDAD OCN MENOS DE 20k DATOS PARA EL ANALISSI PERO EN LA COLUMNA ARTIST HAY UN IDENTIFICADOR, COMO MIRAR SI ME SIRVE PARA USAR MAS DATOS?

PROBBLEMA: serialización / deserialización de datos
df_music['artist'] = df_clean['artist'].apply(lambda x: x['name'] if isinstance(x, dict) else x) # NO DEVUELVE EL RESULTADO ESPERADO
👉 La API devuelve dict
👉 pero al guardar en CSV → se convierte en string
SOLUCION: primero convertir ese string en dict real

In [ ]:
import ast

def clean_artist(x):
    try:
        if isinstance(x, str) and x.startswith('{'):
            x = ast.literal_eval(x)
        return x.get('name') if isinstance(x, dict) else x
    except:
        return None
    
def clean_streamable(x):
    try:
        if isinstance(x, str) and x.startswith('{'):
            x = ast.literal_eval(x)
        return x.get('#text') if isinstance(x, dict) else x
    except:
        return None

In [ ]:
df_clean['artist'] = df_clean['artist'].apply(clean_artist)
df_clean['streamable'] = df_clean['streamable'].apply(clean_streamable)


In [ ]:
df_music['artist'] = df_clean['artist'].apply(lambda x: x['name'] if isinstance(x, dict) else x)
df_music['streamable'] = df_clean['streamable'].apply(lambda x: x['#text'] if isinstance(x, dict) else x)
df_music.head()

,name,duration,playcount,listeners,streamable,artist,country,genre_tag
0,Stateside + Zara Larsson,176,14787193.0,1013640.0,0,PinkPantheress,NaN,NaN
1,Body to Body,189,2615165.0,250542.0,0,BTS,NaN,NaN
2,Hooligan,182,2367820.0,233389.0,0,BTS,NaN,NaN
3,Swim,159,6050938.0,231653.0,0,BTS,NaN,NaN
4,FYA,180,2324099.0,227425.0,0,BTS,NaN,NaN


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 550, in _run_callback
    f = callback(*args, **kwargs)
        ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vscode/.local/lib/python3.11/site-packages/ipykernel/iostream.py", line 105, in _handle_subscription
    event_type = frame[0]
                 ~~~~~^^^
IndexError: index out of range
ERROR:tornado.general:Uncaught exception in zmqstream callback
Traceback (most recent call last):
  File "/home/vscode/.local/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 600, in _handle_events
    self._handle_recv()
  File "/home/vscode/.local/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 629, in _handle_recv
    self._run_callback(callback, msg)
  File "/home/vscode/.local/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 550, in _run_callback
    f = callback(*args, 

> **Observaciones y decisiones:**
> * Se eliminaron duplicados por combinación de **name + artist**.
> * Se eliminaron registros sin **playcount** o **listeners**, ya que son variables clave para el análisis.
> * Se rellenaron valores faltantes en variables secundarias como **country** y **genre_tag**.
>
> **Justificación de eliminación de datos**
>
> Se eliminaron registros con valores nulos en variables clave (playcount, listeners) porque:
> 
> * Son esenciales para medir popularidad.
> * Su ausencia impide análisis estadísticos fiables. 
> * Introducen ruido y sesgos en modelos de machine learning.
>
> Este enfoque prioriza calidad sobre cantidad de datos.


### 1. **Análisis del mercado musical (descriptivo):** 



(1) Top canciones / artistas por periodo (tiempo); 

(2) Evolución temporal de popularidad; 

(3) Géneros en crecimiento vs decrecimiento

#### (1) Top canciones / artistas por periodo (tiempo)

## API Request + Cuestiones generales (antes 23/03/2026)

In [ ]:
ret = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=chart.gettoptracks&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret.json()
for track in ret.json()["tracks"]["track"]:
  print(track["name"], ". Duracion:", track["duration"],"minutos")
music = pd.DataFrame(ret.json()["tracks"]["track"])
music.head()
# Cleaning the data frame:
music.drop(['mbid', 'image', 'url'],axis=1)


#### Filtrando ds:

* Nombre y duracion de las canciones
* Comparación de la canción por titulo (happy or sad)
* Duración promedio de los top30 tracks


##### Nombre y duración de las canciones


In [ ]:
for track in ret.json()["tracks"]["track"]:
  print(track["name"], ". Duracion:", track["duration"],"minutos")


##### Comparación de grupos (canciones 'happy'-'sad'):

In [ ]:
df_happy = music[music['name'].str.contains('happy',case=False,na=False)]
df_happy.count()
df_sad = music[music['name'].str.contains('sad', case=False, na=False)]
df_sad.count()


##### Duración promedio de top30 tracks:

In [ ]:
df_top30_duration = music.sort_values(by='playcount', ascending=False) # .head(30)
df_top30_duration.iloc[:30]
df_top30_duration.head()

df_top30_duration.loc[:, 'duration'] = pd.to_numeric(df_top30_duration['duration'], errors='coerce')
mean = df_top30_duration['duration'].mean()
mean


#### Procedencia de los artisitas top10:
 
* Oyentes de artists top10

**Info api request:**

In [ ]:
ret_artist = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=geo.gettopartists&country=spain&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret_artist.json()
artists_country = pd.DataFrame(ret_artist.json()["topartists"]["artist"])
artists_country.head()


**Oyentes de los artistas top 10:**

In [ ]:
df_top10 = music.head(10)
df_top10['duration'] = pd.to_numeric(df_top10['duration'], errors='coerce')
df_top10['listeners'] = pd.to_numeric(df_top10['listeners'], errors='coerce')

df_top10['artist_name'] = df_top10['artist'].apply(lambda x: x['name'])

plt.figure(figsize = (15, 5))

plt.bar(df_top10['artist_name'], df_top10['listeners'], color = ['skyblue'])
plt.xlabel('Artist name')
plt.ylabel('Listeners count')
plt.title("Listener of top 10 artist")
plt.show()


NameError: name 'music' is not defined

#### Géneros musicales en tendencia global

In [ ]:

def get_trending_tags(api_key):
    url = f"https://ws.audioscrobbler.com/2.0/?method=tag.getTopTags&api_key={api_key}&format=json"
    response = requests.get(url)
    return response.json()['toptags']['tag']

api_key = '63e059c3c912a3f642daf2372484d183'
get_trending_tags(api_key)
ret_generes = get_trending_tags(api_key)
top_generes = pd.DataFrame(ret_generes)[['name', 'count', 'reach']]
top_generes.head()

# **ML**

1. **Definición del problema:** 

* Regresión → predecir popularidad (0–100)

* Clasificación → hit vs no hit

2. **Modelos básicos (mínimo viable):** (1) Linear Regression; (2) Random Forest; (3) XGBoost / LightGBM

3. **Evaluación:** (1) RMSE / MAE (regresión); (2) Accuracy / F1 (clasificación)

4. **Interpretabilidad:**

* Feature importance

* SHAP values (muy top para destacar)